# Adding a Prior to the Model

The prior is an output module that writes its energy under its own `output_key`, and an `Aggregation` module sums the terms into the total energy. This energy may represent a restraint holding a bond at a given length, a repulsive term the training data never covered, a bias pushing a relaxation somewhere it would not go on its own: see `ZBLRepulsionEnergy`, the Coulomb modules, or `HarmonicBond` in `schnetpack.atomistic`. Once the prior is part of the energy, the response module differentiates it along with everything else, and the forces derived from it pick it up on their own.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from copy import deepcopy

from ase.io import read
from ase.optimize import LBFGS

import schnetpack as spk
from schnetpack import properties
from schnetpack.interfaces.ase_interface import SpkCalculator, AtomsConverter
from schnetpack.utils.compatibility import load_model

We load the force field model and wrap it in a `SpkCalculator`, which evaluates one ASE `Atoms` object per call.

In [ ]:
model_path = "../../tests/testdata/md_ethanol.model"

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model
model = load_model(model_path, device=device)
cutoff = model.representation.cutoff.item()

ENERGY_UNIT = "kcal/mol"
POSITION_UNIT = "Ang"
FMAX = 0.001  # converged once no force exceeds this, in eV/Ang
MAX_STEPS = 1000  # give up after this many optimizer steps


def spk_calculator(model=model_path):
    return SpkCalculator(
        model=model,
        neighbor_list=spk.transform.MatScipyNeighborList(cutoff=cutoff),
        device=device,
        energy_unit=ENERGY_UNIT,
        position_unit=POSITION_UNIT,
    )

The demo below relaxes ethanol's C-O bond starting from a few different conformers.

In [ ]:
input_structure_file = "../../tests/testdata/ethanol_conformers.xyz"

# load the starting structures
conformers = read(input_structure_file, index=":")

In [ ]:
def with_prior(model, prior, prior_key, energy_key=properties.energy):
    """A copy of ``model`` whose energy carries an extra term, forces included.
    """
    model = deepcopy(model)
    modules = list(model.output_modules)

    response = next(
        idx
        for idx, module in enumerate(modules)
        if isinstance(module, (spk.atomistic.Forces, spk.atomistic.Response))
    )
    aggregation = spk.atomistic.Aggregation(
        keys=[energy_key, prior_key], output_key=energy_key
    )
    model.output_modules = nn.ModuleList(
        modules[:response] + [prior, aggregation] + modules[response:]
    )

    # the model caches what its modules require and produce, so both have to be redone
    model.collect_derivatives()
    model.collect_outputs()
    return model

Ethanol comes out of the file as `C C O H H H H H H`, so atoms `(0, 2)` are the C-O bond.
It relaxes to about 1.43 Å on its own; we restrain it to 1.8 Å and relax the same
structures twice, once with the plain model and once with the composed one, each
structure one at a time with ASE's `LBFGS`.

In [ ]:
BOND = (0, 2)  # the C-O bond of ethanol
BOND_KEY = "energy_bond"

restrained_model = with_prior(
    model,
    spk.atomistic.HarmonicBond(
        atom_pair=BOND,
        bond_length=1.8,  # Angstrom, well beyond the equilibrium C-O distance
        force_constant=50.0,  # eV / Angstrom**2
        energy_unit=ENERGY_UNIT,
        position_unit=POSITION_UNIT,
        output_key=BOND_KEY,
    ),
    prior_key=BOND_KEY,
)

print("output modules:", [type(m).__name__ for m in restrained_model.output_modules])
print("model outputs: ", restrained_model.model_outputs)


def relaxed_bond_lengths(structures, model):
    """Relax every structure with ``model``, one at a time, and report its C-O bond length."""
    calculator = spk_calculator(model)
    lengths = []
    for structure in structures:
        atoms = structure.copy()
        atoms.calc = calculator
        LBFGS(atoms, logfile=None).run(fmax=FMAX, steps=MAX_STEPS)
        lengths.append(atoms.get_distance(*BOND))
    return np.array(lengths)


free_bonds = relaxed_bond_lengths(conformers, model_path)
restrained_bonds = relaxed_bond_lengths(conformers, restrained_model)

print(f"\n{'':<14}{'C-O bond [Ang]':>16}")
for label, bonds in [("unrestrained", free_bonds), ("restrained", restrained_bonds)]:
    print(f"{label:<14}{bonds.mean():>16.3f}")
print(f"{'target':<14}{1.8:>16.3f}")